# Traffic Scene Understanding — Exploration Notebook
End-to-end walkthrough: vision → NLP → alert

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

## 1. Vision — Dummy Data & Model

In [ ]:
from vision.model import build_model, LABELS
from vision.dataset import generate_dummy_annotations, build_dataloaders

model = build_model(pretrained=False)
print('Labels:', LABELS)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
train_ann, val_ann = generate_dummy_annotations('../data', n_train=40, n_val=10)
train_loader, val_loader = build_dataloaders(train_ann, val_ann, batch_size=8)

imgs, labels = next(iter(train_loader))
print('Batch shape:', imgs.shape)
print('Labels shape:', labels.shape)

# Visualise a batch
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
for ax, img, lbl in zip(axes.flat, imgs, labels):
    img_np = (img.permute(1,2,0).numpy() * std + mean).clip(0,1)
    ax.imshow(img_np)
    ax.set_title([LABELS[i] for i, v in enumerate(lbl) if v > 0.5], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Metrics Sanity Check

In [ ]:
from vision.metrics import compute_metrics, format_metrics

preds = torch.randn(32, 4)
targets = torch.randint(0, 2, (32, 4)).float()
m = compute_metrics(preds, targets)
print(format_metrics(m))

## 3. Grad-CAM (untrained model)

In [ ]:
from vision.gradcam import GradCAM, overlay_heatmap

target_layer = model.backbone.layer4[-1].conv2
gradcam = GradCAM(model, target_layer)

dummy_img = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
from torchvision import transforms
t = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
inp = t(dummy_img).unsqueeze(0)

cam, class_idx = gradcam.generate(inp)
overlay = overlay_heatmap(np.array(dummy_img), cam)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,4))
ax1.imshow(dummy_img); ax1.set_title('Original'); ax1.axis('off')
ax2.imshow(overlay); ax2.set_title(f'Grad-CAM: {LABELS[class_idx]}'); ax2.axis('off')
plt.show()

## 4. NLP — spaCy Preprocessing

In [ ]:
from nlp.preprocess import parse_document, extract_keywords, get_dependency_triples

text = 'A major accident occurred on NH-8 near Sector 62 causing heavy traffic congestion.'
doc = parse_document(text)

print('Sentences:', doc.sentences)
print('Noun chunks:', doc.noun_chunks)
print('Named entities:', doc.named_entities)
print('Keywords:', extract_keywords(text))
print('Dep triples:', get_dependency_triples(text))

## 5. Synthetic NLP Data

In [ ]:
from nlp.dataset import generate_ner_sample, generate_cls_samples, CLS_ID2LABEL

print('=== NER Sample ===')
s = generate_ner_sample()
for tok, tag in zip(s.tokens, s.ner_tags):
    print(f'  {tok:<15} {tag}')

print('\n=== Classification Samples ===')
texts, labels = generate_cls_samples(n_per_class=2)
for t, l in zip(texts, labels):
    print(f'  [{CLS_ID2LABEL[l]}] {t}')

## 6. Alert Generation (Template Mode)

In [ ]:
from alerts.generator import generate_alert

scenarios = [
    ('accident', 'Sector 62', 'major', ['rain', 'congestion']),
    ('jam', 'NH-8', 'heavy', ['congestion']),
    ('road_closure', 'MG Road', None, ['clear']),
    ('normal', 'Ring Road', None, ['clear']),
]

for inc, loc, sev, vis in scenarios:
    r = generate_alert(inc, loc, sev, vis, force_template=True)
    print(r['alert_text'])